In [6]:
library(dplyr)
library(readr)
library(stringr)
library(purrr)
library(tibble)


#base_dir <- "/mnt/gsdata/projects/panops/panops-data-registry/data/flux/fluxnet_2017_2025_V02"
base_dir <- "/mnt/gsdata/projects/panops/panops-data-registry/data/flux/Flux4Daniel"

all_zip_files <- list.files(
  base_dir,
  pattern = "\\.zip$",
  recursive = TRUE,
  full.names = TRUE
)

cat("Zip files found:", length(all_zip_files), "\n")

zip_index_list <- list()
bad_zips <- character()

for (z in all_zip_files) {
  
  files_inside <- tryCatch(
    unzip(z, list = TRUE)$Name,
    error = function(e) {
      bad_zips <<- c(bad_zips, z)
      return(NULL)
    }
  )
  
  if (is.null(files_inside)) next
  
  hh_file <- files_inside[
    str_detect(files_inside, "FLUXNET_FLUXMET_HH") &
      str_detect(files_inside, "\\.csv$")
  ]
  
  if (length(hh_file) > 0) {
    zip_index_list[[length(zip_index_list) + 1]] <- tibble(
      zip_path = z,
      hh_file = hh_file
    )
  }
}

zip_index <- bind_rows(zip_index_list) %>%
  mutate(
    site_id = str_extract(hh_file, "[A-Z]{2}-[A-Za-z0-9]{3}")
  ) %>%
  filter(!is.na(site_id)) %>%
  distinct(site_id, zip_path, hh_file, .keep_all = TRUE)

cat("Sites with FLUXMET HH:", n_distinct(zip_index$site_id), "\n")
cat("Bad/corrupted zips:", length(bad_zips), "\n")

head(zip_index)

Zip files found: 717 
Sites with FLUXMET HH: 694 
Bad/corrupted zips: 15 


zip_path,hh_file,site_id
<chr>,<chr>,<chr>
/mnt/gsdata/projects/panops/panops-data-registry/data/flux/Flux4Daniel/AMF_AR-Bal_FLUXNET_2012-2013_v1.3_r1.zip,AMF_AR-Bal_FLUXNET_FLUXMET_HH_2012-2013_v1.3_r1.csv,AR-Bal
/mnt/gsdata/projects/panops/panops-data-registry/data/flux/Flux4Daniel/AMF_AR-CCa_FLUXNET_2012-2020_v1.3_r1.zip,AMF_AR-CCa_FLUXNET_FLUXMET_HH_2012-2020_v1.3_r1.csv,AR-CCa
/mnt/gsdata/projects/panops/panops-data-registry/data/flux/Flux4Daniel/AMF_AR-CCg_FLUXNET_2018-2024_v1.3_r1.zip,AMF_AR-CCg_FLUXNET_FLUXMET_HH_2018-2024_v1.3_r1.csv,AR-CCg
/mnt/gsdata/projects/panops/panops-data-registry/data/flux/Flux4Daniel/AMF_AR-TF1_FLUXNET_2016-2018_v1.3_r1.zip,AMF_AR-TF1_FLUXNET_FLUXMET_HH_2016-2018_v1.3_r1.csv,AR-TF1
/mnt/gsdata/projects/panops/panops-data-registry/data/flux/Flux4Daniel/AMF_AR-TF2_FLUXNET_2016-2018_v1.3_r1.zip,AMF_AR-TF2_FLUXNET_FLUXMET_HH_2016-2018_v1.3_r1.csv,AR-TF2
/mnt/gsdata/projects/panops/panops-data-registry/data/flux/Flux4Daniel/AMF_BR-CST_FLUXNET_2014-2015_v1.3_r1.zip,AMF_BR-CST_FLUXNET_FLUXMET_HH_2014-2015_v1.3_r1.csv,BR-CST


In [7]:
library(dplyr)
library(readr)
library(stringr)
library(purrr)
library(tibble)

zip_path <- "Flux4Daniel"

extract_elev_from_zip <- function(zip_path, site_id) {
  
  files_inside <- unzip(zip_path, list = TRUE)$Name
  
  bif_file <- files_inside[
    str_detect(files_inside, "BIF") &
      str_detect(files_inside, "\\.csv$")
  ][1]
  
  if (is.na(bif_file)) {
    return(tibble(SITE_ID = site_id, LOCATION_ELEV = NA_real_))
  }
  
  bif <- read_csv(
    unz(zip_path, bif_file),
    show_col_types = FALSE
  )
  
  elev <- bif %>%
    filter(VARIABLE == "LOCATION_ELEV") %>%
    pull(DATAVALUE) %>%
    as.numeric()
  
  tibble(
    SITE_ID = site_id,
    LOCATION_ELEV = elev[1]
  )
}

In [8]:
elev_table <- map2_dfr(
  zip_index$zip_path,
  zip_index$site_id,
  extract_elev_from_zip
)

head(elev_table)
summary(elev_table$LOCATION_ELEV)

SITE_ID,LOCATION_ELEV
<chr>,<dbl>
AR-Bal,130
AR-CCa,83
AR-CCg,84
AR-TF1,40
AR-TF2,60
BR-CST,468


   Min. 1st Qu.  Median    Mean 3rd Qu.    Max.    NA's 
-9999.0    61.0   260.0   367.5   547.0  4146.0      81 

In [12]:
library(dplyr)

library(readr)
meta <- read_csv(

  file.path(base_dir, "selected_site_metadata.csv"),

  show_col_types = FALSE

)



In [13]:
names(meta)
names(elev_table)

[1] "data_hub"               "SITE_ID"                "site_name"             
 [4] "location_lat"           "location_long"          "igbp"                  
 [7] "network"                "team_member_name"       "team_member_role"      
[10] "team_member_email"      "first_year"             "last_year"             
[13] "download_link"          "fluxnet_product_name"   "product_citation"      
[16] "product_id"             "oneflux_code_version"   "product_source_network"

[1] "SITE_ID"       "LOCATION_ELEV"

In [14]:
meta_with_elev <- meta %>%

  left_join(

    elev_table,

    by = "SITE_ID"

  )

write_csv(

  meta_with_elev,

  file.path(base_dir, "fluxnet_site_metadata_clean_with_elevation.csv")

)

summary(meta_with_elev$LOCATION_ELEV)

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max.    NA's 
-9999.0    61.0   260.0   367.5   547.0  4146.0     104 